In [19]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [35]:
from pathlib import Path

data_path = Path("/content/drive/MyDrive/ASL_Project/asl_dataset/asl_dataset")

print("Dataset root:", data_path)
print("\nFolders inside dataset root:")

for folder in sorted(data_path.iterdir()):
    if folder.is_dir():
        print(folder.name, "->", len(list(folder.iterdir())), "items")

Dataset root: /content/drive/MyDrive/ASL_Project/asl_dataset/asl_dataset

Folders inside dataset root:
0 -> 70 items
1 -> 70 items
2 -> 70 items
3 -> 70 items
4 -> 70 items
5 -> 70 items
6 -> 70 items
7 -> 70 items
8 -> 70 items
9 -> 70 items
a -> 70 items
b -> 70 items
c -> 70 items
d -> 70 items
e -> 70 items
f -> 70 items
g -> 70 items
h -> 70 items
i -> 70 items
j -> 70 items
k -> 70 items
l -> 70 items
m -> 70 items
n -> 70 items
o -> 70 items
p -> 70 items
q -> 70 items
r -> 70 items
s -> 70 items
t -> 65 items
u -> 70 items
v -> 70 items
w -> 70 items
x -> 70 items
y -> 70 items
z -> 70 items


Data Preprocessing and *Augmentation*

In [36]:
from torchvision import datasets, transforms

# Training images
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomRotation(10),
    transforms.RandomResizedCrop(224, scale=(0.9, 1.0)),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# Validation images
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

Dataset Loading and Train/Validation Split

In [37]:
from pathlib import Path
from torchvision import datasets
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import train_test_split

raw = Path("/content/drive/MyDrive/ASL_Project/asl_dataset/asl_dataset")

train_data = datasets.ImageFolder(raw, transform=train_transform)
val_data = datasets.ImageFolder(raw, transform=val_transform)

assert train_data.classes == val_data.classes

class_names = train_data.classes
num_classes = len(class_names)

In [38]:
indices = list(range(len(train_data)))
targets = train_data.targets

train_indices, val_indices = train_test_split(
    indices,
    test_size=0.2,
    random_state=42,
    stratify=targets
)

train_dataset = Subset(train_data, train_indices)
val_dataset = Subset(val_data, val_indices)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False
)

print("Class names:", class_names)
print("Number of classes:", num_classes)
print("Total images:", len(train_data))
print("Training images:", len(train_dataset))
print("Validation images:", len(val_dataset))

Class names: ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']
Number of classes: 36
Total images: 2515
Training images: 2012
Validation images: 503


VGG16 Transfer Learning Model[link text](https://)

In [39]:
import torch
import torch.nn as nn
from torchvision import models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load pretrained VGG16
model = models.vgg16(
    weights=models.VGG16_Weights.DEFAULT
)

model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    model.classifier.parameters(),
    lr=0.0001
)

print("Device:", device)
print("Number of classes:", num_classes)
print("Class names:", class_names)

Device: cuda
Number of classes: 36
Class names: ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']


VGG16 Model Training

In [40]:
import time

epochs = 3
start_time = time.time()

for epoch in range(epochs):
    model.train()
    running_loss = 0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)

    print(
        f"Epoch [{epoch+1}/{epochs}] "
        f"Loss: {running_loss/total:.4f} "
        f"Accuracy: {100*correct/total:.2f}%"
    )

training_time = time.time() - start_time
print("Total training time:", round(training_time, 2), "seconds")

Epoch [1/3] Loss: 2.8192 Accuracy: 32.55%
Epoch [2/3] Loss: 0.7060 Accuracy: 76.84%
Epoch [3/3] Loss: 0.3998 Accuracy: 87.18%
Total training time: 351.4 seconds


VGG16 Evaluation

In [43]:
model.eval()

correct = 0
total = 0
val_loss = 0.0

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        val_loss += loss.item() * images.size(0)
        predictions = outputs.argmax(dim=1)

        correct += (predictions == labels).sum().item()
        total += labels.size(0)

print("Validation Loss:", round(val_loss / total, 4))
print("Validation Accuracy:", round(100 * correct / total, 2), "%")
print("Training Time:", round(training_time, 2), "seconds")

Validation Loss: 0.3162
Validation Accuracy: 90.26 %
Training Time: 351.4 seconds


Save Trained VGG16 Model

In [44]:
from pathlib import Path
import torch

save_path = Path("/content/drive/MyDrive/ASL_Project/results")
save_path.mkdir(parents=True, exist_ok=True)

torch.save({
    "model_state_dict": model.state_dict(),
    "classes": class_names,
    "num_classes": num_classes
}, save_path / "vgg16_asl.pth")

print("Model saved:", save_path / "vgg16_asl.pth")
print("Class labels:", class_names)

Model saved: /content/drive/MyDrive/ASL_Project/results/vgg16_asl.pth
Class labels: ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']
